# Additional Lecture 3 — Hands-on Workshop
### *Augment, Generate & **Evaluate** across Modalities* (Lecture 3 of 3)

**Course:** DA631E — Artificial Intelligence for Data Science · Malmö University
**Lecturer:** Ezequiel López-Rubio · **Date:** Friday, 25 September 2026

You now have **Lecture 7 (Validation & Evaluation)**, so today we do not just *peek* — we use
**train/test splits, macro-F1, TSTR, and leakage checks** to decide whether generated/augmented
data actually helps.

This notebook **runs end-to-end** and produces real numbers. Then it is **your turn**: change the
data (prompts, mix, balance, filtering) and see the metric move. Small shared **real** test sets are
embedded so your evaluation is honest.

---
### Before you start
1. **Enable the free GPU:** *Runtime → Change runtime type → T4 GPU*.
2. Run the setup, then do **at least two tracks** spanning **≥ 2 modalities**.
3. Keep it light; one model at a time; free memory between sections.

**The evaluation toolkit we use (from Lecture 7):** split *before* anything · augment/generate the
*training* side only · report **macro-F1** · **TSTR** for synthetic data · check for **leakage**.


## 0 · Setup + a shared text generator

In [1]:
!pip -q install "transformers>=4.44" accelerate sentencepiece scikit-learn librosa

import gc, torch, pandas as pd, numpy as np

def free_memory(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("GPU available:", torch.cuda.is_available())

GPU available: True


In [2]:
# One chat model powers text augmentation (Track A) and tabular generation (Track B).
from transformers import pipeline

gen = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct",
               torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
               device_map="auto")

def chat(user, system="You are a helpful assistant.", temp=0.9, max_new=80):
    msgs = [{"role": "system", "content": system},
            {"role": "user",   "content": user}]
    out = gen(msgs, max_new_tokens=max_new, do_sample=True, temperature=temp, top_p=0.95)
    return out[0]["generated_text"][-1]["content"].strip()

print(chat("Say hello in one short sentence."))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Hello! How can I assist you today?


## Track A · Text: rescue an imbalanced set, then **evaluate**

**Scenario.** Support tickets are imbalanced: many *billing*, few *bug*. We LLM-augment **only the
training** minority class and measure **macro-F1** on a fixed **real** test set (never augmented).

In [3]:
# --- data: an imbalanced TRAIN pool + a balanced, held-out REAL test set ---
train_billing = [
    "I was charged twice for my subscription this month.",
    "Can I get an invoice for my last payment?",
    "My refund has not arrived after two weeks.",
    "Why did the monthly price increase without notice?",
    "Please update the credit card on my account.",
    "I want to cancel and get a prorated refund.",
    "The discount code did not apply at checkout.",
    "You billed me after I already cancelled.",
    "I need a receipt for my company expenses.",
]
train_bug = [                                   # minority class (only 3)
    "The app crashes whenever I upload a photo.",
    "The login button does nothing on Android.",
    "Search results never load, just a spinner.",
]
train_texts  = train_billing + train_bug
train_labels = ["billing"] * len(train_billing) + ["bug"] * len(train_bug)

real_test = [
    ("I was double-billed and need a refund.",              "billing"),
    ("Your invoice total does not match my plan.",          "billing"),
    ("Please remove the extra charge from June.",           "billing"),
    ("The subscription renewed at the wrong price.",        "billing"),
    ("The export button throws an error every time.",       "bug"),
    ("The app freezes on the payment screen.",              "bug"),
    ("Notifications stopped working after the update.",      "bug"),
    ("The page is blank when I open my dashboard.",         "bug"),
]
test_texts, test_labels = [t for t, _ in real_test], [l for _, l in real_test]
print("train:", pd.Series(train_labels).value_counts().to_dict(),
      "| test:", pd.Series(test_labels).value_counts().to_dict())

train: {'billing': 9, 'bug': 3} | test: {'billing': 4, 'bug': 4}


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, classification_report

def macro_f1(tr_texts, tr_labels):
    clf = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
    clf.fit(tr_texts, tr_labels)
    pred = clf.predict(test_texts)
    return f1_score(test_labels, pred, average="macro"), pred

base_f1, _ = macro_f1(train_texts, train_labels)
print("baseline macro-F1:", round(base_f1, 3))

baseline macro-F1: 0.333


In [5]:
# Augment ONLY the minority ("bug") training items with LLM paraphrases.
def paraphrase(text, n=3):
    out = chat(f"Paraphrase this software bug report in {n} different short ways, "
               f"one per line, keeping it a BUG report: '{text}'",
               system="You rewrite text. Return only the rewrites, one per line.",
               temp=1.1, max_new=120)
    return [ln.strip("-*0123456789. ").strip() for ln in out.split("\n") if ln.strip()][:n]

aug_texts, aug_labels = [], []
for t in train_bug:
    for v in paraphrase(t, n=3):
        if v:
            aug_texts.append(v); aug_labels.append("bug")

print("added", len(aug_texts), "synthetic bug examples")
aug_f1, _ = macro_f1(train_texts + aug_texts, train_labels + aug_labels)
print("baseline :", round(base_f1, 3))
print("augmented:", round(aug_f1, 3), "  (did it improve?)")

[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


added 9 synthetic bug examples
baseline : 0.333
augmented: 0.733   (did it improve?)


**Your turn (Track A).** Inspect `aug_texts` — did any paraphrase drift away from being a *bug*?
Remove drifted items and re-run. Try `n=5`, or a higher `temperature`. Does more always help?

## Track B · Tabular: generate from a schema, then **TSTR**

**Scenario.** Prototype a churn model with little real data. Generate a synthetic table, then
**Train on Synthetic, Test on Real** and report macro-F1 + a confusion matrix.

In [6]:
import json, re

def _to_int(v):
    m = re.search(r"-?\d+", str(v))          # tolerate ints given as strings, e.g. "34"
    return int(m.group()) if m else None

def _to_bool(v):
    if isinstance(v, bool):
        return v
    return str(v).strip().lower() in {"true", "yes", "1", "y", "t"}

def gen_record():
    p = ('Generate ONE fictional customer as a single JSON object with EXACTLY '
         'these keys and no other text:\n'
         '{"age": <int 18-80>, "tenure_months": <int 0-72>, '
         '"tickets": <int 0-15>, "churned": <true or false>}\n'
         'Example: {"age": 34, "tenure_months": 12, "tickets": 2, "churned": false}')
    raw = chat(p, system="You output a single valid JSON object and nothing else.",
               max_new=96, temp=0.7)
    m = re.search(r"\{.*?\}", raw, flags=re.S)                 # first {...} block
    if not m:
        return None
    s = m.group().replace("True", "true").replace("False", "false").replace("'", '"')
    try:
        rec = json.loads(s)
        age, ten, tik = _to_int(rec.get("age")), _to_int(rec.get("tenure_months")), _to_int(rec.get("tickets"))
        if None in (age, ten, tik):
            return None
        if not (18 <= age <= 80 and 0 <= ten <= 72 and 0 <= tik <= 15):
            return None
        return {"age": age, "tenure_months": ten, "tickets": tik,
                "churned": _to_bool(rec.get("churned"))}
    except Exception:
        return None

# Collect valid rows, capping attempts so the cell can never hang.
synth, attempts = [], 0
while len(synth) < 30 and attempts < 60:
    r = gen_record()
    if r:
        synth.append(r)
    attempts += 1
print(f"LLM produced {len(synth)} valid rows from {attempts} attempts")

# Safety net: if the small model under-delivers, top up so TSTR can still run.
if len(synth) < 20:
    import random
    random.seed(0)
    need = 20 - len(synth)
    print(f"topping up with {need} programmatic rows so the track can proceed")
    for _ in range(need):
        churn = random.random() < 0.5
        synth.append({"age": random.randint(18, 80),
                      "tenure_months": random.randint(0, 18) if churn else random.randint(19, 72),
                      "tickets": random.randint(5, 15) if churn else random.randint(0, 4),
                      "churned": churn})

# Explicit columns => the frame always has the schema, even if empty (no KeyError).
synth_df = pd.DataFrame(synth, columns=["age", "tenure_months", "tickets", "churned"])
synth_df["churned"] = synth_df["churned"].astype(bool)
print("total synthetic rows:", len(synth_df))
synth_df.head()

[transformers] Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

LLM produced 30 valid rows from 30 attempts
total synthetic rows: 30


,age,tenure_months,tickets,churned
0,32,6,1,True
1,34,12,2,False
2,29,6,3,True
3,42,9,3,True
4,25,5,3,True


In [7]:
# a small, hand-written REAL held-out test set (churn if short tenure & many tickets)
real_df = pd.DataFrame([
    (24,  2, 9, True), (55, 60, 0, False), (31,  8, 6, True), (44, 40, 1, False),
    (28,  3, 8, True), (60, 65, 0, False), (37, 15, 5, True), (49, 33, 2, False),
    (23,  1, 7, True), (52, 50, 1, False), (34, 10, 4, True), (41, 28, 0, False),
], columns=["age", "tenure_months", "tickets", "churned"])

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix

feat = ["age", "tenure_months", "tickets"]
clf = RandomForestClassifier(n_estimators=100, random_state=0)
clf.fit(synth_df[feat], synth_df["churned"])            # TRAIN on synthetic
pred = clf.predict(real_df[feat])                        # TEST on real
print("TSTR macro-F1:", round(f1_score(real_df["churned"], pred, average="macro"), 3))
print("confusion matrix [rows=true F/T]:\n", confusion_matrix(real_df["churned"], pred))

TSTR macro-F1: 0.333
confusion matrix [rows=true F/T]:
 [[0 6]
 [0 6]]


In [8]:
free_memory(gen)   # release the LLM before the image/audio models

**Your turn (Track B).** Are the synthetic value ranges realistic? Do `tickets`/`tenure` relate to
`churned` the way the *real* set does? Fix the prompt to inject that structure and watch TSTR change.

## Track C · Image **or** Audio: end-to-end (pick one)

Generate a tiny **2-class** set, then **train + evaluate** a simple classifier. We hold out some
generated items as the test split (ideal: a few *real* items). Run **one** of the two cells below.

In [9]:
# ---- OPTION 1: IMAGE (apple vs banana) ----
!pip -q install diffusers
from diffusers import AutoPipelineForText2Image
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

t2i = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
t2i = t2i.to("cuda" if torch.cuda.is_available() else "cpu")

classes = {"apple":  "a photo of a single red apple on a white background",
           "banana": "a photo of a single yellow banana on a white background"}
imgs, labs = [], []
for label, prompt in classes.items():
    for _ in range(5):                                   # 2 x 5 = 10 images
        imgs.append(t2i(prompt, num_inference_steps=1, guidance_scale=0.0).images[0])
        labs.append(label)
free_memory(t2i)

def color_hist(pil):                                     # cheap 48-D feature
    a = np.asarray(pil.resize((64, 64))) / 255.0
    return np.concatenate([np.histogram(a[..., k], bins=16, range=(0, 1))[0] for k in range(3)])

X = np.array([color_hist(im) for im in imgs]); yv = np.array(labs)
Xtr, Xte, ytr, yte = train_test_split(X, yv, test_size=0.4, stratify=yv, random_state=0)
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print("image accuracy:", round(accuracy_score(yte, clf.predict(Xte)), 3))
print(confusion_matrix(yte, clf.predict(Xte)))

[transformers] `Siglip2ImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Siglip2ImageProcessor` instead.
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


model_index.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

image accuracy: 0.5
[[2 0]
 [2 0]]


In [10]:
# ---- OPTION 2: AUDIO (yes vs no, across several speakers) ----
!pip -q install datasets soundfile sentencepiece
from transformers import pipeline
from datasets import load_dataset
import librosa
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

tts = pipeline("text-to-speech", model="microsoft/speecht5_tts")
try:
    emb = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
    speakers = [torch.tensor(emb[i]["xvector"]).unsqueeze(0) for i in (1000, 3000, 5000, 7306, 200)]
except Exception as e:
    print("x-vector set unavailable, using pseudo-random voices:", e)
    speakers = []
    for s in range(5):
        g = torch.Generator().manual_seed(s); v = torch.randn(1, 512, generator=g)
        speakers.append(v / v.norm())

feats, labs = [], []
for word in ["yes", "no"]:
    for spk in speakers:                                 # within-class variety = different voices
        out = tts(word, forward_params={"speaker_embeddings": spk})
        y = np.asarray(out["audio"], dtype="float32"); sr = out["sampling_rate"]
        feats.append(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13).mean(axis=1))
        labs.append(word)
free_memory(tts)

X = np.array(feats); yv = np.array(labs)
Xtr, Xte, ytr, yte = train_test_split(X, yv, test_size=0.4, stratify=yv, random_state=0)
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print("audio accuracy:", round(accuracy_score(yte, clf.predict(Xte)), 3))

config.json:   0%|          | 0.00/2.06k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  585MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  585MB            

model.safetensors: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 50.6MB            

README.md:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

model.safetensors: downloading bytes:           |  0.00B            

cmu-arctic-xvectors.py:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

x-vector set unavailable, using pseudo-random voices: Dataset scripts are no longer supported, but found cmu-arctic-xvectors.py
audio accuracy: 0.5


**Your turn (Track C).** For images, raise the count per class or add a third class. For audio,
add more speakers/words. Testing on *held-out generated* items is optimistic — how would you get a few
**real** test items, and would the accuracy survive?

## Track D · Quality & **leakage** audit

Be the skeptic. Measure **near-duplicates**, **diversity**, and — crucially — **train/test leakage**,
which silently inflates scores. Here we audit the Track A text sets.

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

all_train = train_texts + aug_texts
V = TfidfVectorizer().fit(all_train + test_texts)

# 1) near-duplicate rate within the (augmented) training set
S = cosine_similarity(V.transform(all_train)); np.fill_diagonal(S, 0)
print("near-duplicate share:", round(float((S > 0.8).any(axis=1).mean()), 3))

# 2) diversity: type-token ratio
words = " ".join(all_train).lower().split()
print("type-token ratio   :", round(len(set(words)) / max(len(words), 1), 3))

# 3) LEAKAGE: any training item almost identical to a test item?
L = cosine_similarity(V.transform(all_train), V.transform(test_texts))
print("leaked train items :", int((L > 0.9).any(axis=1).sum()), "(should be 0)")

near-duplicate share: 0.0
type-token ratio   : 0.703
leaked train items : 0 (should be 0)


**Your turn (Track D).** If near-duplicate share is high, your set is *bigger* but not more
*informative*. Deduplicate (drop rows with cosine > 0.9 to another) and re-run Track A — does macro-F1 hold up?

## Track E · Data-centric mini-challenge

**Goal:** using *only* data you augment/generate, maximise **macro-F1** on the shared **real** test set.
The model is **fixed** for everyone (TF-IDF + Logistic Regression) — the **data** is the only variable.

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score

# shared REAL, human-written test set (do NOT train on this)
real_test_sent = pd.DataFrame([
    ("Absolutely love it, works better than expected.", "positive"),
    ("Best purchase I have made this year.",            "positive"),
    ("Comfortable, well built, and great value.",       "positive"),
    ("Fast delivery and the quality is excellent.",     "positive"),
    ("Stopped working after three days, very upset.",   "negative"),
    ("Cheaply made and overpriced, avoid.",             "negative"),
    ("Support ignored my emails for a week.",           "negative"),
    ("It arrived broken and refunds are a nightmare.",  "negative"),
], columns=["text", "label"])

def score(train_df):
    clf = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
    clf.fit(train_df["text"], train_df["label"])
    return f1_score(real_test_sent["label"], clf.predict(real_test_sent["text"]), average="macro")

# ---- a weak starter training set: improve it! ----
my_train = pd.DataFrame([
    ("good product", "positive"), ("i like it", "positive"),
    ("bad product", "negative"), ("i hate it", "negative"),
], columns=["text", "label"])
print("starter macro-F1:", round(score(my_train), 3))

starter macro-F1: 0.333


**Your turn (Track E).** Re-load the `gen` model (Section 0) and *generate* a diverse, balanced
training set with an attribute grid (see Lecture 2), filter duplicates/drift, then rebuild `my_train`
and re-run `score(my_train)`. What lifts the number most — **size, balance, or diversity**?

## Reflection & deliverable

**Discuss.** Where did generated/augmented data help — and hurt? Which quality problem was hardest to
detect? Did leakage fool anyone? Would you trust synthetic data for a **privacy-sensitive** Swedish/EU
project, and — given Lecture 7 — *how* would you prove to a stakeholder that it works?

**Submit** your notebook with: (1) at least **two** tracks across **≥ 2 modalities**; (2) a proper
**evaluation** for each (split, macro-F1, and one leakage/quality check); (3) a ~150-word reflection.

*Assessed on sound **evaluation** and insight — not on topping the leaderboard.*

---
**Connections:** Lecture 7 gave you today's evaluation discipline · Lectures 8–9 (supervised
learning) are the models you feed · Lecture 12 (neural nets) and Lecture 13 (what is *inside* the
generators) · use these tools in your **mini-projects**.